# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset loaded:")
print(metadata["name"])
print("Description:")
print(metadata["description"])

## 2. Data Overview

Review available record sets, fields, and their IDs.

The Croissant schema organizes data as record sets containing fields and columns. Each entity is referenced by its unique `@id`.

Let's inspect the available record sets to identify their structure.

In [ ]:
# Examine all available record sets in the dataset
record_sets = dataset.record_sets()
print("Record Sets and their @id values:")
for record_set in record_sets:
    print(f"RecordSet name: {record_set.name}, @id: {record_set.id}")

# For demonstration, inspect fields and columns for each record set
for record_set in record_sets:
    fields = record_set.fields()
    print(f"\nFields for RecordSet '{record_set.name}' (@id: {record_set.id}):")
    for field in fields:
        print(f"  Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
        columns = field.columns()
        if columns:
            for col in columns:
                print(f"    Column name: {col.name}, @id: {col.id}")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Entities are referenced by their `@id`.

Here, we pull the data for each record set using their `@id` and preview the columns.

In [ ]:
# Gather the @id of all record sets
record_set_ids = [rs.id for rs in dataset.record_sets()]

# Load data from all record sets into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nDataFrame for RecordSet @id: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())

# Example: Display columns and head for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nFirst RecordSet @id: {first_rs_id}")
    print(dataframes[first_rs_id].columns.tolist())
    print(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, as an example, we:
- Filter by a numerical column (e.g., 'Age' field referenced via its @id)
- Normalize that column
- Group records by a categorical field (e.g., 'Sex')

**Note:** To ensure consistent referencing, we use the field and column `@id`s determined from the overview above.

In [ ]:
# Choose the RecordSet @id that contains clinical data
# For this example, we search for a field with data_type 'Integer' or 'Float' (e.g., Age)

# Detect a numeric field
numeric_field_id = None
record_set_id = None
for rs in dataset.record_sets():
    for field in rs.fields():
        if field.data_type in ['Integer', 'Float'] and ('age' in field.name.lower() or 'Age' in field.name):
            numeric_field_id = field.id
            record_set_id = rs.id
            print(f"Selected numeric field: {field.name} (@id: {field.id}) in RecordSet @id: {rs.id}")
            break
    if numeric_field_id:
        break

# Detect a group field, e.g., Sex
group_field_id = None
if record_set_id is not None:
    for field in dataset.get_record_set(record_set_id).fields():
        if field.data_type == 'Text' and ('sex' in field.name.lower() or 'Sex' in field.name):
            group_field_id = field.id
            print(f"Selected group field: {field.name} (@id: {field.id})")
            break

# Conduct filtering, normalization, and grouping
if record_set_id and numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    threshold = 50  # Example threshold, e.g., Age > 50
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. We'll create a histogram for the selected numeric field (e.g., Age), and a boxplot grouped by the categorical field (e.g., Sex), referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizations
if record_set_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR^2 dataset using `mlcroissant`. Key steps included:
- Loading metadata and tabular data using Croissant URLs
- Identifying record sets, fields, and columns by their `@id`
- Extracting data into DataFrames and referencing entities with `@id`
- Filtering and normalizing numeric fields, grouping by categorical attributes
- Visualizing data distributions and relationships

This process supports reproducible, FAIR-aligned data analysis and helps reveal clinicopathological patterns in second primary colorectal cancer survivors.